In [2]:
pip install shap

   ---------------------------------------- 0.0/549.1 kB ? eta -:--:--
   -------------------------------------- - 524.3/549.1 kB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 549.1/549.1 kB 2.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.8 MB 4.5 MB/s eta 0:00:01
   -------------------------- ------------- 1.8/2.8 MB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 4.8 MB/s  0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   -- ------------------------------------- 2.6/38.1 MB 12.3 MB/s eta 0:00:03
   ------ --------------------------------- 5.8/38.1 MB 13.3 MB/s eta 0:00:03
   --------- ------------------------------ 9.4/38.1 MB 14.3 MB/s eta 0:00:03
   ------------- -------------------------- 12.8/38.1 MB 15.1 MB/s eta 0:00:02
   ----------------- ---------------------- 17.0/38.1 MB 15.9 MB/s eta 0:00:02
   -------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\athar\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [3]:
"""
Extract Real Evidence for Conversational Interface

This notebook extracts actual clinical data to show WHY each agent made its prediction:
- Agent 1: Top lab values + SHAP contributions
- Agent 2: Key clinical note phrases
- Agent 3: Vital sign statistics + SHAP contributions
"""

import numpy as np
import pandas as pd
import joblib
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
import shap

# Disease list
DISEASE_LIST = [
    'SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE',
    'ACUTE_KIDNEY_INJURY', 'HEART_FAILURE', 'ATRIAL_FIBRILLATION',
    'CORONARY_ARTERY_DISEASE', 'ANEMIA', 'PANCREATITIS'
]

print("="*70)
print("🔬 EXTRACTING REAL EVIDENCE DATA")
print("="*70)

🔬 EXTRACTING REAL EVIDENCE DATA


In [4]:
# Load existing patient database
with open('../../interface_data/patient_database.json', 'r') as f:
    patient_database = json.load(f)

print(f"✅ Loaded {len(patient_database)} patients")

# Get test indices
test_patient_ids = [p['patient_id'] for p in patient_database]
test_indices = [p['test_index'] for p in patient_database]

print(f"✅ Test patient IDs: {min(test_patient_ids)} to {max(test_patient_ids)}")

✅ Loaded 1393 patients
✅ Test patient IDs: 6 to 4642


In [14]:

print("\n" + "="*70)
print("🔬 LOADING AGENT 1 LAB DATA (VALIDATION SET)")
print("="*70)

# Load VALIDATION lab features (not the full dataset!)
labs_df = pd.read_csv('../../data/processed/lab_features_val.csv')
print(f"✅ Loaded validation lab data: {labs_df.shape}")

# Get lab column names (everything except SUBJECT_ID)
LAB_COLUMNS = [col for col in labs_df.columns if col != 'SUBJECT_ID']
print(f"✅ Found {len(LAB_COLUMNS)} lab features")

# Extract arrays
X_labs = labs_df[LAB_COLUMNS].values
labs_subject_ids = labs_df['SUBJECT_ID'].values

print(f"✅ Lab features array: {X_labs.shape}")
print(f"✅ SUBJECT_IDs: {len(labs_subject_ids)} patients")

LABS_AVAILABLE = True


🔬 LOADING AGENT 1 LAB DATA (VALIDATION SET)
✅ Loaded validation lab data: (4675, 156)
✅ Found 155 lab features
✅ Lab features array: (4675, 155)
✅ SUBJECT_IDs: 4675 patients


In [15]:

print("\n" + "="*70)
print("📝 LOADING AGENT 2 DISCHARGE SUMMARIES")
print("="*70)

# Load discharge summaries
notes_df = pd.read_csv('../../data/processed/discharge_summaries_all.csv')
print(f"✅ Loaded discharge summaries: {notes_df.shape}")
print(f"   Total summaries: {len(notes_df):,}")

notes_subject_ids = notes_df['SUBJECT_ID'].values

NOTES_AVAILABLE = True


📝 LOADING AGENT 2 DISCHARGE SUMMARIES
✅ Loaded discharge summaries: (59652, 3)
   Total summaries: 59,652


In [10]:

print("\n" + "="*70)
print("💓 LOADING AGENT 3 VITAL SIGNS")
print("="*70)

# Load vital sequences
vital_sequences = np.load('../../data/processed/vital_sequences.npy')
vital_patient_ids = np.load('../../data/processed/vital_sequence_patient_ids.npy')

print(f"✅ Loaded vital sequences: {vital_sequences.shape}")
print(f"   (patients, timesteps, vitals)")

VITAL_NAMES = ['Heart_Rate', 'Systolic_BP', 'Diastolic_BP', 'MAP', 'Resp_Rate', 'Temperature', 'SpO2']

VITALS_AVAILABLE = True


💓 LOADING AGENT 3 VITAL SIGNS
✅ Loaded vital sequences: (23130, 24, 7)
   (patients, timesteps, vitals)


In [11]:

print("\n" + "="*70)
print("🔗 MATCHING TEST PATIENTS TO ORIGINAL DATA")
print("="*70)

# Load patient database to get test patient IDs
with open('../../interface_data/patient_database.json', 'r') as f:
    patient_database = json.load(f)

test_patient_ids = [p['patient_id'] for p in patient_database]
print(f"✅ Test set: {len(test_patient_ids)} patients")

# Find indices in original data
# For labs
labs_test_mask = labs_df['SUBJECT_ID'].isin(test_patient_ids)
labs_test_df = labs_df[labs_test_mask].copy()
print(f"\n📊 Agent 1 (Labs):")
print(f"   Test patients found: {len(labs_test_df)} / {len(test_patient_ids)}")

# For notes (multiple notes per patient, take first discharge summary per patient)
notes_test_df = notes_df[notes_df['SUBJECT_ID'].isin(test_patient_ids)].copy()
# Keep only one note per patient (first one)
notes_test_df = notes_test_df.drop_duplicates(subset='SUBJECT_ID', keep='first')
print(f"\n📊 Agent 2 (Notes):")
print(f"   Test patients found: {len(notes_test_df)} / {len(test_patient_ids)}")

# For vitals
vitals_test_mask = np.isin(vital_patient_ids, test_patient_ids)
vitals_test_indices = np.where(vitals_test_mask)[0]
print(f"\n📊 Agent 3 (Vitals):")
print(f"   Test patients found: {len(vitals_test_indices)} / {len(test_patient_ids)}")

# Create lookup dictionaries
labs_lookup = {row['SUBJECT_ID']: idx for idx, row in labs_test_df.iterrows()}
notes_lookup = {row['SUBJECT_ID']: row['TEXT'] for _, row in notes_test_df.iterrows()}
vitals_lookup = {vital_patient_ids[i]: i for i in vitals_test_indices}

print(f"\n✅ Created lookup dictionaries for all agents")


🔗 MATCHING TEST PATIENTS TO ORIGINAL DATA
✅ Test set: 1393 patients

📊 Agent 1 (Labs):
   Test patients found: 543 / 1393

📊 Agent 2 (Notes):
   Test patients found: 1096 / 1393

📊 Agent 3 (Vitals):
   Test patients found: 577 / 1393

✅ Created lookup dictionaries for all agents


In [16]:

print("\n" + "="*70)
print("🔗 LOADING SUBJECT_ID MAPPING")
print("="*70)

# Load fusion SUBJECT_IDs
fusion_subject_ids = np.load('../../data/processed/fusion_subject_ids.npy')
print(f"✅ Loaded fusion SUBJECT_IDs: {len(fusion_subject_ids)} patients")

# Load fusion data
X_fusion = np.load('../../data/processed/X_fusion_val.npy')
print(f"✅ Loaded X_fusion_val: {X_fusion.shape}")

# Verify they match
assert len(fusion_subject_ids) == len(X_fusion), "Mismatch between SUBJECT_IDs and fusion data!"

# Create test split (same as in your other notebooks)
indices = np.arange(len(X_fusion))
train_val_idx, test_idx = train_test_split(indices, test_size=0.30, random_state=42)

# Get test SUBJECT_IDs
test_subject_ids = fusion_subject_ids[test_idx]
print(f"\n✅ Test set: {len(test_idx)} patients")
print(f"   Test SUBJECT_IDs range: {test_subject_ids.min()} to {test_subject_ids.max()}")

# Match to original data sources
print("\n" + "="*70)
print("🔍 MATCHING TEST PATIENTS TO ORIGINAL DATA")
print("="*70)

# For labs
labs_test_mask = labs_df['SUBJECT_ID'].isin(test_subject_ids)
labs_test_df = labs_df[labs_test_mask].copy()
print(f"\n📊 Agent 1 (Labs):")
print(f"   Test patients found: {len(labs_test_df)} / {len(test_subject_ids)}")

# For notes (one note per patient)
notes_test_df = notes_df[notes_df['SUBJECT_ID'].isin(test_subject_ids)].copy()
notes_test_df = notes_test_df.drop_duplicates(subset='SUBJECT_ID', keep='first')
print(f"\n📊 Agent 2 (Notes):")
print(f"   Test patients found: {len(notes_test_df)} / {len(test_subject_ids)}")

# For vitals
vitals_test_mask = np.isin(vital_patient_ids, test_subject_ids)
vitals_test_indices = np.where(vitals_test_mask)[0]
print(f"\n📊 Agent 3 (Vitals):")
print(f"   Test patients found: {len(vitals_test_indices)} / {len(test_subject_ids)}")

# Create lookup dictionaries
labs_lookup = {row['SUBJECT_ID']: idx for idx, row in labs_test_df.iterrows()}
notes_lookup = {row['SUBJECT_ID']: row['TEXT'] for _, row in notes_test_df.iterrows()}
vitals_lookup = {vital_patient_ids[i]: i for i in vitals_test_indices}

print(f"\n✅ Created lookup dictionaries")

# Update patient_database with correct SUBJECT_IDs
print("\n" + "="*70)
print("🔄 UPDATING PATIENT DATABASE WITH SUBJECT_IDs")
print("="*70)

for idx, patient in enumerate(patient_database):
    # Map test_index to actual SUBJECT_ID
    test_idx_pos = patient['test_index']
    actual_subject_id = int(test_subject_ids[test_idx_pos])
    
    # Update patient_id to be the actual SUBJECT_ID
    patient['patient_id'] = actual_subject_id

print(f"✅ Updated {len(patient_database)} patient records with actual SUBJECT_IDs")
print(f"   SUBJECT_ID range: {min(p['patient_id'] for p in patient_database)} to {max(p['patient_id'] for p in patient_database)}")


🔗 LOADING SUBJECT_ID MAPPING
✅ Loaded fusion SUBJECT_IDs: 4643 patients
✅ Loaded X_fusion_val: (4643, 33)

✅ Test set: 1393 patients
   Test SUBJECT_IDs range: 61.0 to 99992.0

🔍 MATCHING TEST PATIENTS TO ORIGINAL DATA

📊 Agent 1 (Labs):
   Test patients found: 1393 / 1393

📊 Agent 2 (Notes):
   Test patients found: 1359 / 1393

📊 Agent 3 (Vitals):
   Test patients found: 1036 / 1393

✅ Created lookup dictionaries

🔄 UPDATING PATIENT DATABASE WITH SUBJECT_IDs
✅ Updated 1393 patient records with actual SUBJECT_IDs
   SUBJECT_ID range: 61 to 99992


In [17]:

print("\n" + "="*70)
print("🔬 EXTRACTING AGENT 1 EVIDENCE")
print("="*70)

from tqdm.auto import tqdm

# Lab feature names (parsed from column names)
# Format: lab_51301_mean → extract the stat type and map to readable name
LAB_NAMES = {
    'lab_51301': 'WBC', 'lab_51300': 'WBC_manual', 'lab_51221': 'Hematocrit',
    'lab_51222': 'Hemoglobin', 'lab_51265': 'Platelets', 'lab_51277': 'RDW',
    'lab_50912': 'Creatinine', 'lab_50971': 'Potassium', 'lab_50983': 'Sodium',
    'lab_50902': 'Chloride', 'lab_50882': 'Bicarbonate', 'lab_50893': 'Calcium',
    'lab_50931': 'Glucose', 'lab_50960': 'Magnesium', 'lab_50970': 'Phosphate',
    'lab_50878': 'AST', 'lab_50861': 'ALT', 'lab_50863': 'Alkaline_Phosphatase',
    'lab_50885': 'Total_Bilirubin', 'lab_50862': 'Albumin', 'lab_51006': 'BUN',
    'lab_50813': 'Lactate', 'lab_50820': 'pH', 'lab_50821': 'PaO2',
    'lab_50818': 'PaCO2', 'lab_51491': 'Temperature', 'lab_51003': 'Troponin'
}

# Normal ranges
NORMAL_RANGES = {
    'Creatinine': (0.7, 1.3, 'mg/dL'), 'BUN': (7, 20, 'mg/dL'),
    'WBC': (4.0, 11.0, 'K/μL'), 'Hemoglobin': (12.0, 16.0, 'g/dL'),
    'Hematocrit': (36, 46, '%'), 'Platelets': (150, 400, 'K/μL'),
    'Sodium': (136, 145, 'mmol/L'), 'Potassium': (3.5, 5.0, 'mmol/L'),
    'Chloride': (98, 107, 'mmol/L'), 'Bicarbonate': (22, 28, 'mmol/L'),
    'Glucose': (70, 100, 'mg/dL'), 'Lactate': (0.5, 2.0, 'mmol/L'),
    'Calcium': (8.5, 10.5, 'mg/dL'), 'Magnesium': (1.7, 2.2, 'mg/dL'),
    'Phosphate': (2.5, 4.5, 'mg/dL'), 'Albumin': (3.5, 5.5, 'g/dL'),
    'Total_Bilirubin': (0.1, 1.2, 'mg/dL'), 'ALT': (7, 56, 'U/L'),
    'AST': (10, 40, 'U/L'), 'Alkaline_Phosphatase': (44, 147, 'U/L'),
    'pH': (7.35, 7.45, ''), 'PaCO2': (35, 45, 'mmHg'),
    'PaO2': (75, 100, 'mmHg'), 'Temperature': (36.5, 37.5, '°C')
}

def parse_lab_name(col_name):
    """Parse lab_51301_mean -> (WBC, mean)"""
    parts = col_name.split('_')
    if len(parts) >= 3:
        lab_code = f"{parts[0]}_{parts[1]}"
        stat = parts[2]
        lab_name = LAB_NAMES.get(lab_code, lab_code)
        return f"{lab_name}_{stat}"
    return col_name

def get_agent1_evidence(subject_id, disease, top_n=5):
    """Extract top lab evidence for a patient"""
    
    # Find patient in labs_df
    if subject_id not in labs_lookup:
        return []
    
    patient_row_idx = labs_lookup[subject_id]
    patient_labs = labs_df.loc[patient_row_idx, LAB_COLUMNS].values.reshape(1, -1)
    
    # Load model
    model_path = f'../../models/agent1/agent1_{disease.lower()}.joblib'
    model = joblib.load(model_path)
    
    # Compute SHAP
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(patient_labs)
    
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    
    feature_importance = np.abs(shap_values[0])
    
    # Get top features
    top_indices = np.argsort(feature_importance)[-top_n:][::-1]
    
    evidence = []
    for idx in top_indices:
        col_name = LAB_COLUMNS[idx]
        lab_value = patient_labs[0, idx]
        
        if np.isnan(lab_value):
            continue
        
        # Parse lab name
        parsed_name = parse_lab_name(col_name)
        base_name = parsed_name.split('_')[0] if '_' in parsed_name else parsed_name
        
        # Get normal range
        normal_range = NORMAL_RANGES.get(base_name, (None, None, ''))
        low, high, unit = normal_range
        
        # Determine status
        status = '✓'
        if low is not None and high is not None:
            if lab_value < low:
                status = '↓'
            elif lab_value > high:
                status = '↑'
        
        evidence.append({
            'lab_name': parsed_name,
            'value': float(lab_value),
            'unit': unit,
            'normal_low': low,
            'normal_high': high,
            'status': status,
            'importance': float(feature_importance[idx]),
            'shap_value': float(shap_values[0][idx])
        })
    
    return evidence

# Extract for all patients
print("Extracting Agent 1 evidence (this may take 5-10 minutes)...")

for patient in tqdm(patient_database, desc="Agent 1 Evidence"):
    subject_id = patient['patient_id']
    
    patient['agent1_evidence'] = {}
    
    for disease in DISEASE_LIST:
        try:
            evidence = get_agent1_evidence(subject_id, disease, top_n=5)
            patient['agent1_evidence'][disease] = evidence
        except Exception as e:
            patient['agent1_evidence'][disease] = []

print("✅ Agent 1 evidence extraction complete!")


🔬 EXTRACTING AGENT 1 EVIDENCE
Extracting Agent 1 evidence (this may take 5-10 minutes)...


Agent 1 Evidence:   0%|          | 0/1393 [00:00<?, ?it/s]

✅ Agent 1 evidence extraction complete!


In [18]:

print("\n" + "="*70)
print("📝 EXTRACTING AGENT 2 EVIDENCE")
print("="*70)

# Disease-specific keywords
DISEASE_KEYWORDS = {
    'SEPSIS': ['sepsis', 'septic', 'infection', 'fever', 'hypotension', 'tachycardia', 'lactate', 'antibiotics', 'bacteremia', 'systemic inflammatory response'],
    'PNEUMONIA': ['pneumonia', 'infiltrate', 'consolidation', 'cough', 'sputum', 'respiratory', 'crackles', 'chest x-ray', 'lung', 'pulmonary'],
    'RESPIRATORY_FAILURE': ['respiratory failure', 'intubated', 'ventilator', 'hypoxia', 'dyspnea', 'oxygen', 'mechanical ventilation', 'ards', 'hypoxemic'],
    'ACUTE_KIDNEY_INJURY': ['acute kidney injury', 'aki', 'creatinine', 'renal failure', 'dialysis', 'oliguria', 'azotemia', 'kidney', 'renal'],
    'HEART_FAILURE': ['heart failure', 'chf', 'pulmonary edema', 'dyspnea', 'volume overload', 'diuresis', 'ejection fraction', 'cardiomyopathy', 'cardiac'],
    'ATRIAL_FIBRILLATION': ['atrial fibrillation', 'afib', 'a-fib', 'irregular rhythm', 'rate control', 'anticoagulation', 'cardioversion', 'arrhythmia'],
    'CORONARY_ARTERY_DISEASE': ['coronary artery disease', 'cad', 'chest pain', 'angina', 'myocardial infarction', 'stent', 'catheterization', 'ischemia', 'cardiac'],
    'ANEMIA': ['anemia', 'hemoglobin', 'transfusion', 'blood loss', 'hematocrit', 'iron deficiency', 'low blood count'],
    'PANCREATITIS': ['pancreatitis', 'pancreatic', 'amylase', 'lipase', 'abdominal pain', 'epigastric', 'pancreas']
}

def extract_key_phrases(note_text, disease, max_phrases=3):
    """Extract key phrases from clinical note"""
    if pd.isna(note_text) or note_text == '':
        return []
    
    note_lower = str(note_text).lower()
    keywords = DISEASE_KEYWORDS[disease]
    
    found_phrases = []
    
    # Find sentences containing keywords
    sentences = note_lower.split('.')
    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) < 20:  # Skip very short fragments
            continue
        
        for keyword in keywords:
            if keyword in sentence and len(sentence) < 200:  # Reasonable length
                # Capitalize first letter
                clean_sentence = sentence[0].upper() + sentence[1:] if len(sentence) > 1 else sentence
                if clean_sentence not in found_phrases:
                    found_phrases.append(clean_sentence)
                if len(found_phrases) >= max_phrases:
                    return found_phrases[:max_phrases]
    
    # If not enough full sentences, just note keyword presence
    if len(found_phrases) < 2:
        for keyword in keywords[:3]:  # Top 3 keywords
            if keyword in note_lower:
                found_phrases.append(f"Mentions: {keyword}")
                if len(found_phrases) >= max_phrases:
                    break
    
    return found_phrases[:max_phrases]

# Extract for all patients
print("Extracting Agent 2 evidence...")

for patient in tqdm(patient_database, desc="Agent 2 Evidence"):
    subject_id = patient['patient_id']
    
    patient['agent2_evidence'] = {}
    
    # Get note text
    note_text = notes_lookup.get(subject_id, '')
    
    for disease in DISEASE_LIST:
        phrases = extract_key_phrases(note_text, disease, max_phrases=3)
        patient['agent2_evidence'][disease] = phrases

print("✅ Agent 2 evidence extraction complete!")


📝 EXTRACTING AGENT 2 EVIDENCE
Extracting Agent 2 evidence...


Agent 2 Evidence:   0%|          | 0/1393 [00:00<?, ?it/s]

✅ Agent 2 evidence extraction complete!


In [19]:

print("\n" + "="*70)
print("💓 EXTRACTING AGENT 3 EVIDENCE")
print("="*70)

VITAL_NAMES = ['Heart_Rate', 'Systolic_BP', 'Diastolic_BP', 'MAP', 'Resp_Rate', 'Temperature', 'SpO2']

def compute_vital_statistics(patient_vital_idx):
    """Compute statistics from 24-hour vital sequence"""
    sequence = vital_sequences[patient_vital_idx]  # Shape: (24, 7)
    
    stats = []
    for i, vital_name in enumerate(VITAL_NAMES):
        vital_values = sequence[:, i]
        
        # Remove NaN values
        valid_values = vital_values[~np.isnan(vital_values)]
        
        if len(valid_values) == 0:
            continue
        
        stats.append({
            'vital_name': vital_name,
            'mean': float(np.mean(valid_values)),
            'std': float(np.std(valid_values)),
            'min': float(np.min(valid_values)),
            'max': float(np.max(valid_values))
        })
    
    return stats

def get_agent3_evidence(subject_id, disease, top_n=5):
    """Extract top vital sign evidence"""
    
    # Find patient in vitals
    if subject_id not in vitals_lookup:
        return []
    
    vital_idx = vitals_lookup[subject_id]

    vital_stats = compute_vital_statistics(vital_idx)

    evidence = []
    
    for stat in vital_stats[:top_n]:

        evidence.append({
            'vital_name': stat['vital_name'],
            'mean': stat['mean'],
            'std': stat['std'],
            'min': stat['min'],
            'max': stat['max'],
            'importance': float(stat['std'])
        })
    
    return evidence

# Extract for all patients
print("Extracting Agent 3 evidence...")

for patient in tqdm(patient_database, desc="Agent 3 Evidence"):
    subject_id = patient['patient_id']
    
    patient['agent3_evidence'] = {}
    
    for disease in DISEASE_LIST:
        try:
            evidence = get_agent3_evidence(subject_id, disease, top_n=5)
            patient['agent3_evidence'][disease] = evidence
        except Exception as e:
            patient['agent3_evidence'][disease] = []

print("✅ Agent 3 evidence extraction complete!")


💓 EXTRACTING AGENT 3 EVIDENCE
Extracting Agent 3 evidence...


Agent 3 Evidence:   0%|          | 0/1393 [00:00<?, ?it/s]

✅ Agent 3 evidence extraction complete!


In [22]:

print("\n" + "="*70)
print("📊 EXTRACTING EVIDENCE (SIMPLIFIED APPROACH)")
print("="*70)

# Lab names mapping
LAB_NAME_MAP = {
    'lab_51301_mean': 'WBC', 'lab_51222_mean': 'Hemoglobin', 
    'lab_51265_mean': 'Platelets', 'lab_50912_mean': 'Creatinine',
    'lab_51006_mean': 'BUN', 'lab_50971_mean': 'Potassium',
    'lab_50983_mean': 'Sodium', 'lab_50902_mean': 'Chloride',
    'lab_50931_mean': 'Glucose', 'lab_50813_mean': 'Lactate',
    'lab_50885_mean': 'Total_Bilirubin', 'lab_50878_mean': 'AST',
    'lab_50861_mean': 'ALT', 'lab_50862_mean': 'Albumin'
}

NORMAL_RANGES = {
    'WBC': (4.0, 11.0, 'K/μL'), 'Hemoglobin': (12.0, 16.0, 'g/dL'),
    'Platelets': (150, 400, 'K/μL'), 'Creatinine': (0.7, 1.3, 'mg/dL'),
    'BUN': (7, 20, 'mg/dL'), 'Potassium': (3.5, 5.0, 'mmol/L'),
    'Sodium': (136, 145, 'mmol/L'), 'Chloride': (98, 107, 'mmol/L'),
    'Glucose': (70, 100, 'mg/dL'), 'Lactate': (0.5, 2.0, 'mmol/L'),
    'Total_Bilirubin': (0.1, 1.2, 'mg/dL'), 'AST': (10, 40, 'U/L'),
    'ALT': (7, 56, 'U/L'), 'Albumin': (3.5, 5.5, 'g/dL')
}

def get_simple_lab_evidence(subject_id):
    """Get lab values without SHAP"""
    if subject_id not in labs_lookup:
        return []
    
    row_idx = labs_lookup[subject_id]
    evidence = []
    
    for col_name, readable_name in list(LAB_NAME_MAP.items())[:7]:  # Top 7 labs
        if col_name in labs_df.columns:
            value = labs_df.loc[row_idx, col_name]
            
            if pd.notna(value):
                normal = NORMAL_RANGES.get(readable_name, (None, None, ''))
                low, high, unit = normal
                
                status = '✓'
                if low and high:
                    if value < low:
                        status = '↓'
                    elif value > high:
                        status = '↑'
                
                evidence.append({
                    'lab_name': readable_name,
                    'value': float(value),
                    'unit': unit,
                    'normal_low': low,
                    'normal_high': high,
                    'status': status
                })
    
    return evidence

def get_simple_note_evidence(subject_id, disease):
    """Get clinical note phrases"""
    if subject_id not in notes_lookup:
        return []
    
    note_text = notes_lookup[subject_id]
    if pd.isna(note_text):
        return []
    
    note_lower = str(note_text).lower()
    keywords = DISEASE_KEYWORDS.get(disease, [])
    
    phrases = []
    for keyword in keywords[:3]:
        if keyword in note_lower:
            phrases.append(f"Mentions: {keyword}")
    
    return phrases[:3]

def get_simple_vital_evidence(subject_id):
    """Get vital statistics"""
    if subject_id not in vitals_lookup:
        return []
    
    vital_idx = vitals_lookup[subject_id]
    sequence = vital_sequences[vital_idx]
    
    evidence = []
    for i, vital_name in enumerate(VITAL_NAMES):
        values = sequence[:, i]
        valid = values[~np.isnan(values)]
        
        if len(valid) > 0:
            evidence.append({
                'vital_name': vital_name,
                'mean': float(np.mean(valid)),
                'std': float(np.std(valid)),
                'min': float(np.min(valid)),
                'max': float(np.max(valid))
            })
    
    return evidence

# Extract for all patients
print("\n🔬 Extracting Agent 1 evidence...")
for patient in tqdm(patient_database, desc="Agent 1"):
    subject_id = patient['patient_id']
    # Same evidence for all diseases (just showing lab values)
    evidence = get_simple_lab_evidence(subject_id)
    patient['agent1_evidence'] = {disease: evidence for disease in DISEASE_LIST}

print("\n📝 Extracting Agent 2 evidence...")
for patient in tqdm(patient_database, desc="Agent 2"):
    subject_id = patient['patient_id']
    patient['agent2_evidence'] = {}
    for disease in DISEASE_LIST:
        patient['agent2_evidence'][disease] = get_simple_note_evidence(subject_id, disease)

print("\n💓 Extracting Agent 3 evidence...")
for patient in tqdm(patient_database, desc="Agent 3"):
    subject_id = patient['patient_id']
    evidence = get_simple_vital_evidence(subject_id)
    patient['agent3_evidence'] = {disease: evidence for disease in DISEASE_LIST}

print("\n✅ Evidence extraction complete!")

# SAVE IMMEDIATELY
print("\n💾 Saving patient database...")
with open('../../interface_data/patient_database.json', 'w') as f:
    json.dump(patient_database, f, indent=2)

print("✅ Saved!")

# Verify
print("\n🔍 Verifying...")
sample = patient_database[0]
print(f"Patient {sample['patient_id']}:")
print(f"  Agent 1: {len(sample['agent1_evidence']['SEPSIS'])} items")
print(f"  Agent 2: {len(sample['agent2_evidence']['SEPSIS'])} items")
print(f"  Agent 3: {len(sample['agent3_evidence']['SEPSIS'])} items")


📊 EXTRACTING EVIDENCE (SIMPLIFIED APPROACH)

🔬 Extracting Agent 1 evidence...


Agent 1:   0%|          | 0/1393 [00:00<?, ?it/s]


📝 Extracting Agent 2 evidence...


Agent 2:   0%|          | 0/1393 [00:00<?, ?it/s]


💓 Extracting Agent 3 evidence...


Agent 3:   0%|          | 0/1393 [00:00<?, ?it/s]


✅ Evidence extraction complete!

💾 Saving patient database...
✅ Saved!

🔍 Verifying...
Patient 16926:
  Agent 1: 7 items
  Agent 2: 0 items
  Agent 3: 7 items
